# Idea 1e: does any single aligned-count rule give no wrong rescue on ALL sets?
After Idea 1d (fresh NUDT valid1313: 3 rescues, 1 wrong), the threshold and margin are scanned over all five low-confidence dumps (official150, NUDT valid1212/valid1313, fresh707/fresh1010), C 0.05 and T 20 fixed. Reading: a setting is usable only if it has 0 wrong rescues on every set AND a net gain on the sets overall. Planned falsification: if every setting that removes the wrong rescue also removes nearly all right ones, idea 1 is dead outside the official dev photos.
The sets are no longer independent for choosing the rule (valid1313 has now been seen), so a pick here is a candidate to be tested on one more unseen set, not a result.


In [ ]:
import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"
from train.checks import replay
from fine_matching.loftr_matcher import reliable_mask
from image_processing.map_levels import level_gsd
SETS = ["official150", "nudt_valid1212", "nudt_valid1313", "nudt_fresh707", "nudt_fresh1010"]


In [ ]:
SIZE = 224
def z_of(photo, level, c, t):
    """For every standard query and candidate: (z, x, y, inliers) with z the excess count of aligned matches over chance."""
    gsd = level_gsd(level); out = []
    for qi in replay.query_ids(photo, replay.STANDARD):
        q = photo["queries"][qi]; reliable = reliable_mask(q["inside"], q["border_px"])
        for k, m in enumerate(q["matches"]):
            if m is None or q["peaks"][k][1] > 1e8:
                continue
            a, b, conf = m["points_a"], m["points_b"], m["confidence"]
            cols = np.clip(np.round(a[:, 0]).astype(int), 0, reliable.shape[1] - 1); rows = np.clip(np.round(a[:, 1]).astype(int), 0, reliable.shape[0] - 1)
            chosen = reliable[rows, cols] & (conf >= c)
            n = int(chosen.sum()); gap = np.hypot(b[chosen, 0] - a[chosen, 0], b[chosen, 1] - a[chosen, 1]) * gsd
            near = int((gap <= t).sum()); chance = n * min(1.0, math.pi * (t / gsd) ** 2 / (SIZE * SIZE))
            out.append(((near - chance) / math.sqrt(chance + 1.0), q["peaks"][k][1] + q["shifts"][k][0], q["peaks"][k][2] + q["shifts"][k][1], float(q["inliers"][k]), qi))
    return out


In [ ]:
def rescue(cands, photo, thr, margin, radius_sigma=2.0):
    """Answer at the candidate with the highest z if z >= thr and no candidate farther than radius_sigma from it has z within `margin` of it."""
    if not cands:
        return None
    z = np.array([c[0] for c in cands]); best = int(np.argmax(z))
    if z[best] < thr:
        return None
    far = [c[0] for c in cands if math.hypot(c[1] - cands[best][1], c[2] - cands[best][2]) / photo["sigma"] > radius_sigma]
    if far and max(far) > z[best] - margin:
        return None
    return cands[best][1], cands[best][2]



In [ ]:
C, T = 0.05, 20.0
firsts, zs = {}, {}
for tag in SETS:
    low = replay.load_set(T7 / "rich_dump" / ("lowconf_" + tag), with_matches=True)
    for pid, photo in low.items():
        first = replay.decide_current(photo, escalate=False)
        entry = dict(tag=tag, pid=pid, points=first["points"], wrong=int(first["answered"] and first["err_sigma"] > 8), answered=first["answered"])
        firsts[(tag, pid)] = entry
        if not first["answered"]:
            zs[(tag, pid)] = (z_of(photo, photo["level"], C, T), photo)
print({t: sum(1 for k in firsts if k[0] == t) for t in SETS}, "skipped:", {t: sum(1 for k in zs if k[0] == t) for t in SETS})


In [ ]:
rows = []
for thr in [10, 12, 15, 20, 30]:
    for margin in [8, 12, 16, 24]:
        for tag in SETS:
            n = sum(1 for k in firsts if k[0] == tag); rescued = wrong = gain = 0
            for (t, pid), (cands, photo) in zs.items():
                if t != tag:
                    continue
                place = rescue(cands, photo, thr, margin)
                if place is not None:
                    err = replay.error_sigma(photo, *place); p = replay.points(err)
                    rescued += 1; wrong += int(err > 8); gain += p
            rows.append(dict(thr=thr, margin=margin, tag=tag, n=n, rescued=rescued, wrong=wrong, gain_per60=gain / n * 6))
G = pd.DataFrame(rows); G.to_csv(T7 / "idea1e_grid.csv", index=False)
S = G.groupby(["thr", "margin"]).agg(rescued=("rescued", "sum"), wrong=("wrong", "sum"), worst_set_gain=("gain_per60", "min"), mean_gain=("gain_per60", "mean")).reset_index()
print(S.sort_values(["wrong", "mean_gain"], ascending=[True, False]).to_string(index=False))
print(G[(G.thr == 10) & (G.margin == 8)][["tag", "rescued", "wrong", "gain_per60"]].to_string(index=False))
